In [ ]:
from datasets import load_dataset
from mne.decoding.csp import CSP
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score, ShuffleSplit, KFold
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.linear_model import LogisticRegression
import numpy as np
from sklearn.utils import resample
from tqdm.auto import tqdm
import json
import pandas as pd
import pickle
import seaborn as sns
sns.set_theme(context='paper', style='white', palette='colorblind', font_scale=.8, font='serif', rc={'font.family': 'serif','figure.figsize': (6.3, 3)})
# Adjust path to import custom modules
import os
import sys
sys.path.append(os.path.abspath(".."))
%load_ext autoreload
%autoreload 2
from utils import get_chan_names, adapt_channel_names_to_mne
import mne
from data.affect_prediction_datamodule import AffectPredictionDataModule
DATASET_PATH = "../data/brain_heart_emotion"
DATA_DIR = "/u/$USER/high_throughput/preproc"
DATA_DIR = "$HOME/projects/datasets/censored/"


# Demonstrate CSP

In [ ]:
from numpy.random import default_rng
import matplotlib.pyplot as plt
rng = default_rng(42)
cov_1 = 1.3
cov_2 = -0.4
cov_mat_1 = np.array([[2, cov_1], [cov_1, 2]])
cov_mat_2 = np.array([[2, cov_2], [cov_2, 2]])
multivariate_sample_1 = rng.multivariate_normal(mean=[0, 0], cov=cov_mat_1, size=2000).T
multivariate_sample_2 = rng.multivariate_normal(mean=[0, 0], cov=cov_mat_2, size=2000).T
plt.scatter(*multivariate_sample_1, label='Sample 1', marker='x')
plt.scatter(*multivariate_sample_2, label='Sample 2', marker='x')
plt.legend()
plt.show()


In [ ]:
segments_1 = np.array(np.split(multivariate_sample_1, 100,axis=1),)
segments_2 = np.array(np.split(multivariate_sample_2, 100, axis=1),)
labels_1 = np.array([0]*100)
labels_2 = np.array([1]*100)
print(segments_1.shape, segments_2.shape)
print(labels_1.shape, labels_2.shape)
X = np.concatenate([segments_1, segments_2])
y = np.concatenate([labels_1, labels_2])
print(X.shape, y.shape)

csp = CSP(n_components=2, transform_into='csp_space')
csp.fit(X, y)
transformed = csp.transform(X)
labels = np.array([0]*2000 + [1]*2000)

transformed = np.concatenate(transformed, axis=1)
plt.scatter(transformed[0][:2000], transformed[1][:2000], label='Population 1', marker='x')
plt.scatter(transformed[0][2000:], transformed[1][2000:], label='Population 2', marker='x')
plt.show()

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(4, 2))

# Plot 1: Scatter plot of multivariate samples
axes[0].scatter(*multivariate_sample_1, marker='x')
axes[0].scatter(*multivariate_sample_2, marker='x')
axes[0].set_xlabel('Fictional EEG Channel 1')
axes[0].set_ylabel('Fictional EEG Channel 2')
axes[0].set_xticks([])
axes[0].set_yticks([])
axes[0].set_title('Data')

# Plot 2: Scatter plot of transformed data
axes[1].scatter(transformed[0][:2000], transformed[1][:2000], label='Population 1', marker='x')
axes[1].scatter(transformed[0][2000:], transformed[1][2000:], label='Population 2', marker='x')
axes[1].set_xlabel('CSP 1')
axes[1].set_ylabel('CSP 2')
axes[1].set_xticks([])
axes[1].set_yticks([])
axes[1].set_title('CSP Transformed Data')
fig.legend(loc='center right', bbox_to_anchor=(1.24, 0.5))


plt.tight_layout()
plt.savefig("../../out/csp_transformed_data.png", dpi=300, bbox_inches='tight')
plt.show()


# Apply on Arousal

In [ ]:
datamodule = AffectPredictionDataModule(dataset_path="../data/brain_heart_emotion", 
                                        data_dir=DATA_DIR, 
                                        num_classes=2,
                                        num_workers=8,)
datamodule.setup()
train_ds_subset = datamodule.train_ds.shuffle().flatten_indices(num_proc=8)#.take(2000)

In [ ]:
train_ds_subset

In [ ]:
def to_single_windows(x):
    return {"signal": x["signal"][:, 0:200], "label": x["label"][0]}
single_epoch_dataset = train_ds_subset.map(to_single_windows, num_proc=6).with_format("numpy")

In [ ]:
single_epoch_val_split = datamodule.val_ds#.take(1000)
single_epoch_val_split = single_epoch_val_split.map(to_single_windows, num_proc=6).with_format("numpy")

single_epoch_test_split = datamodule.test_ds#.take(1000)
single_epoch_test_split = single_epoch_test_split.map(to_single_windows, num_proc=6).with_format("numpy")

In [ ]:
def bootstrap_scores(y_true, y_pred, n_iterations=1000, scorer = balanced_accuracy_score):
    baccs = []
    if len(y_true.shape) != len(y_pred.shape):
        y_pred = y_pred.reshape(y_true.shape)
    yyhat = np.array(list(zip(y_true, y_pred)))
    for i in tqdm(range(n_iterations)):
        resampled = resample(yyhat, replace=True)
        y_true_resampled, y_pred_resampled = resampled[:, 0], resampled[:, 1]
        bacc = scorer(y_true_resampled, y_pred_resampled > 0)
        baccs.append(bacc)
    return np.array(baccs)

In [ ]:
csp = CSP(n_components=4, reg=None, log=True, norm_trace=False)
lda = LDA()
pipeline = make_pipeline(csp, lda)
cv = ShuffleSplit(n_splits=5, test_size=0.2, random_state=42)

results_dict = {
    "percent_of_data": [],
    "mean_accuracy": [],
    "std_accuracy": [],
    "validation_mean_acc": [],
    "validation_std_acc": []
}
for percent_of_data in [0.1, 0.2, 0.4, 0.8, 1.0]:
    n_samples = int(len(single_epoch_dataset) * percent_of_data)
    print(f"Using {n_samples} samples ({percent_of_data*100:.0f}%)")
    subset = single_epoch_dataset.take(n_samples)
    pipeline.fit(subset["signal"].astype(np.float64), subset["label"])
    scores = bootstrap_scores(subset["label"], pipeline.predict(subset["signal"].astype(np.float64)))
    print(f"bootstrapped balanced accuracy: {scores.mean():.3f} ± {scores.std():.3f}")
    validation_scores = bootstrap_scores(pipeline.predict(single_epoch_val_split["signal"].astype(np.float64)), single_epoch_val_split["label"])
    print(f"Validation bootstrapped balanced accuracy: {validation_scores.mean():.3f} ± {validation_scores.std():.3f}")
    results_dict["percent_of_data"].append(percent_of_data)
    results_dict["mean_accuracy"].append(scores.mean())
    results_dict["std_accuracy"].append(scores.std())
    results_dict["validation_mean_acc"].append(validation_scores.mean())
    results_dict["validation_std_acc"].append(validation_scores.std())
with open("../../out/csp_data_scale_study.json", "w") as f:
    json.dump(results_dict, f)




Even though training with all data may not be helpful, sacrificing comparability is not worth training with 20% less data

In [ ]:

scores = {"n_components": [], "train_bacc": [], "val_bacc": {"mean":[], "std":[]}}
for n_components in [2, 4, 6, 8, 12, 16, 24, 32, 60]:
    csp = CSP(n_components=n_components)
    lda = LDA()
    pipeline = make_pipeline(csp, lda)
    pipeline.fit(single_epoch_dataset["signal"].astype(np.float64), single_epoch_dataset["label"])
    train_bacc = balanced_accuracy_score(single_epoch_dataset["label"], pipeline.predict(single_epoch_dataset["signal"].astype(np.float64)) > 0)
    val_bacc_bootstrap = bootstrap_scores(single_epoch_val_split["label"], pipeline.predict(single_epoch_val_split["signal"].astype(np.float64)) > 0)
    print(f"Validation balanced accuracy: {val_bacc_bootstrap.mean():.3f} ± {val_bacc_bootstrap.std():.3f}")
    print('-' * 50)
    scores["n_components"].append(n_components)
    scores["val_bacc"]["mean"].append(np.mean(val_bacc_bootstrap))
    scores["val_bacc"]["std"].append(np.std(val_bacc_bootstrap))
    scores["train_bacc"].append(train_bacc)


In [ ]:
with open("../../out/csp_ncomponent_arousal_scores.json", "w") as f:
    json.dump(scores, f, indent=4)

val_mean = scores["val_bacc"]["mean"]
val_std = scores["val_bacc"]["std"]
scores.pop("val_bacc")
scores_df = pd.DataFrame(scores)
scores_df["val_bacc_mean"] = val_mean
scores_df["val_bacc_std"] = val_std
scores_df.to_csv("../../out/csp_ncomponent_arousal_scores.csv", index=False)
scores_df.head()

In [ ]:
scores_df.rename(columns={"n_components": "CSP Components",
                          "train_bacc": "Same subject validation set Balanced Accuracy",
                      "val_bacc_mean": "Validation Balanced Accuracy",
                      "val_bacc_std": "Validation Balanced Accuracy (std)",
                      "test_bacc": "Test Balanced Accuracy"},
                    inplace=True)
scores_df.to_csv("../../out/csp_arousal_scores.csv", index=False)
scores_df.to_latex("../../out/csp_arousal_scores.tex", index=False, float_format="%.3f")
scores_df

In [ ]:
scores_df = pd.read_csv("../../out/csp_arousal_scores.csv")

## Train with best n_components

In [ ]:
n_components = scores_df.loc[scores_df["Validation Balanced Accuracy"].idxmax(), "CSP Components"].item()
pipeline = make_pipeline(CSP(n_components=n_components), LDA())
pipeline.fit(single_epoch_dataset["signal"].astype(np.float64), single_epoch_dataset["label"])


In [ ]:
train_logits = pipeline.transform(single_epoch_dataset["signal"].astype(np.float64))
val_logits = pipeline.transform(single_epoch_val_split["signal"].astype(np.float64))
test_logits = pipeline.transform(single_epoch_test_split["signal"].astype(np.float64))

np.savez_compressed("../../out/csp_arousal_train_logits.npz", train_logits)
np.savez_compressed("../../out/csp_arousal_val_logits.npz", val_logits)
np.savez_compressed("../../out/csp_arousal_test_logits.npz", test_logits)

In [ ]:

with open("../../out/csp_arousal_pipeline.pkl", "wb") as f:
    pickle.dump(pipeline, f)


### Checkpoint after modelling

In [ ]:
with open("../../out/csp_arousal_pipeline.pkl", "rb") as f:
    pipeline = pickle.load(f)

train_logits = np.load("../../out/csp_arousal_train_logits.npz")["arr_0"]
val_logits = np.load("../../out/csp_arousal_val_logits.npz")["arr_0"]
test_logits = np.load("../../out/csp_arousal_test_logits.npz")["arr_0"]


In [ ]:
print(val_logits[:10])

In [ ]:
from sklearn.metrics import balanced_accuracy_score, roc_auc_score, precision_score, recall_score, f1_score
aurocs = bootstrap_scores(single_epoch_val_split["label"], val_logits, n_iterations=1000, scorer=roc_auc_score)

print(f'Auroc validation: {aurocs.mean():.4f} +/- {aurocs.std():.4f}')

aurocs_test = bootstrap_scores(single_epoch_test_split["label"], test_logits, n_iterations=1000, scorer=roc_auc_score)
print(f'Auroc test: {aurocs_test.mean():.4f} +/- {aurocs_test.std():.4f}')

In [ ]:
metrics_to_calculate = {
    "accuracy": accuracy_score,
    "auroc": roc_auc_score,
    "balanced_accuracy": balanced_accuracy_score,
    "f1": f1_score,
    "precision": precision_score,
    "recall": recall_score,
}
results = {"test": {}, "val": {}}
for metric_name, metric_func in tqdm(list(metrics_to_calculate.items())):
    print(f"Calculating {metric_name}...")
    print(f"\t on validation set")
    results["val"][metric_name] = bootstrap_scores(single_epoch_val_split["label"], val_logits, n_iterations=1000, scorer=metric_func)
    print(f"\t on test set")
    results["test"][metric_name] = bootstrap_scores(single_epoch_test_split["label"], test_logits, n_iterations=1000, scorer=metric_func)
    print(f'\tval {metric_name}: {results["val"][metric_name].mean():.4f} +/- {results["val"][metric_name].std():.4f}')
    print(f'\ttest {metric_name}: {results["test"][metric_name].mean():.4f} +/- {results["test"][metric_name].std():.4f}')


In [ ]:

# turn into a dataframe
results_df = pd.DataFrame()
for metric_name, metric_scores in results["val"].items():
    results_df.loc[metric_name, "val_mean"] = metric_scores.mean()
    results_df.loc[metric_name, "val_std"] = metric_scores.std()
for metric_name, metric_scores in results["test"].items():
    results_df.loc[metric_name, "test_mean"] = metric_scores.mean()
    results_df.loc[metric_name, "test_std"] = metric_scores.std()
results_df.head()

In [ ]:
test_formatted = pd.DataFrame()
for metric in results["test"].keys():
    x = results["test"][metric]
    if not "acc" in metric:
        
        test_formatted[metric] = [f'{x.mean():.4f} ± {x.std():.4f}']
    else:
        test_formatted[metric] = [f'{x.mean()*100:.2f}% ± {x.std()*100:.2f}']

test_formatted

In [ ]:
# add string column with beautiful formatted string
results_df["val"] = results_df.apply(lambda row: f"{row['val_mean']:.4f} ± {row['val_std']:.4f}", axis=1)
results_df["test"] = results_df.apply(lambda row: f"{row['test_mean']:.4f} ± {row['test_std']:.4f}", axis=1)
results_df.to_csv("../../out/csp_arousal_best_model_bootstraps_results.csv", index=True)
results_df.head()

## Interpret masks
parked

In [ ]:
channels = adapt_channel_names_to_mne(get_chan_names(single_epoch_dataset[0]["eeg_channel_indices"]))
montage = mne.channels.make_standard_montage("standard_1020")

info = mne.create_info(channels, sfreq=200, ch_types="eeg")
info.set_montage(montage)

In [ ]:

channels = adapt_channel_names_to_mne(get_chan_names(single_epoch_dataset[0]["eeg_channel_indices"]))
montage = mne.channels.make_standard_montage("standard_1020")

info = mne.create_info(channels, sfreq=200, ch_types="eeg")
info.set_montage(montage)

In [ ]:
lda: LDA = pipeline[1]
lda.coef_

In [ ]:
from visualization.eeg_attribution import attribution_topomap

In [ ]:
weighted_masks = pipeline[0].patterns_[:12].T*lda.coef_[0]
weighted_masks = weighted_masks.T
attribution_topomap(weighted_masks.sum(axis=0), info.ch_names)

In [ ]:
import torch
class csp_lda(torch.nn.Module):
    def __init__(self, pipeline):
        super().__init__()  # Call the constructor of the parent class
        self.pipeline = pipeline
        self.csp_filters = torch.nn.Linear(60, pipeline[0].n_components, bias=False)
        self.lda = torch.nn.Linear(pipeline[0].n_components, 1)
        self.init_weights()
       
    def init_weights(self):
        # Use torch.nn.Parameter to correctly assign weights
        self.csp_filters.weight = torch.nn.Parameter(torch.Tensor(self.pipeline[0].filters_[:self.pipeline[0].n_components]))
        self.lda.weight = torch.nn.Parameter(torch.tensor(self.pipeline[1].coef_[0].reshape(1, -1)))
        self.lda.bias = torch.nn.Parameter(torch.tensor(self.pipeline[1].intercept_))

    def forward(self, x):
        # The forward pass expects log-variance, so we take the log after squaring and averaging
        # x has shape (channels, time_points), linear layer expects (..., features)
        # so we transpose x to (time_points, channels)
        csp_features = torch.log(self.csp_filters(x).square().mean(dim=0))
        lda_features = self.lda(csp_features)
        return lda_features

csp_lda_model = csp_lda(pipeline).to(torch.double)
example = single_epoch_dataset[0]["signal"]
example = torch.tensor(example, dtype=torch.double, requires_grad=True)
csp_lda_model(example.T).backward()
attr = (example*example.grad).detach().numpy()
attribution_topomap(attr.sum(axis=1), info.ch_names)

In [ ]:
pipeline[0].plot_patterns(info=info, sphere="eeglab", image_interp='nearest', contours=0,nrows=3, colorbar=False).show()

In [ ]:
pipeline[0].plot_filters(info=info, image_interp='nearest', sphere='eeglab', contours=0, nrows=3, colorbar=False).show()

In [ ]:
from sklearn.dummy import DummyClassifier
dummy = DummyClassifier(strategy="stratified")
dummy.fit(single_epoch_dataset["signal"].astype(np.float64), single_epoch_dataset["label"])
dummy_scores = bootstrap_scores(single_epoch_val_split["label"], dummy.predict(single_epoch_val_split["signal"].astype(np.float64)), n_iterations=1000, scorer=balanced_accuracy_score)
dummy_test_scores = bootstrap_scores(single_epoch_test_split["label"], dummy.predict(single_epoch_test_split["signal"].astype(np.float64)), n_iterations=1000, scorer=balanced_accuracy_score)
print(f'Dummy classifier validation balanced accuracy: {dummy_scores.mean():.4f} ± {dummy_scores.std():.4f}')
print(f'Dummy classifier test balanced accuracy: {dummy_test_scores.mean():.4f} ± {dummy_test_scores.std():.4f}')